# CSV processing: reading a bank statement

**Before you run anything:** keep `statement.csv` in the *same folder* as this notebook. The code below opens it by name, so if it lives somewhere else you'll get a `FileNotFoundError`.

Run the cells top to bottom. If a cell defines a function, run it before the cells that use it.

This is the same idea as last lesson's regex work, but instead of hunting fields out of raw PDF text by hand, the bank already split them into columns for us. Our job shifts from *finding* the data to *deciding what it means*.


## 1. Look at the raw rows first

Always eyeball the data before computing anything. `csv.DictReader` reads each row into a dict keyed by the header names.


In [ ]:
import csv

with open("statement.csv", newline="", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

print("total rows:", len(rows))
print("columns:", list(rows[0].keys()))
print()
for row in rows[:3]:
    print(row["Posted Date"], "|", row["Amount"], "|", row["Merchant name"] or "(blank)", "|", row["Full description"][:40])

The `utf-8-sig` encoding matters: many bank exports start with a hidden byte-order mark, and without `-sig` the first column name reads as `\ufeffPosted Date` and every lookup on it fails in a way that looks like a typo.

Notice the `Amount` column: debits are wrapped in parentheses like `($86.32)`, credits are plain like `$1,875.40`. The sign lives in the parentheses, not a minus sign.


## 2. Parse the amount

`float("($86.32)")` doesn't return the wrong number, it *raises*. So we strip the `$`, strip thousands commas, and read `(` `)` as negative. This is a spot where plain string methods beat regex: clearer to read, top to bottom.


In [ ]:
def parse_amount(raw: str) -> float:
    s = raw.strip()
    negative = s.startswith("(") and s.endswith(")")
    s = s.strip("()").replace("$", "").replace(",", "").strip()
    return -float(s) if negative else float(s)

# sanity checks
assert parse_amount("($86.32)") == -86.32
assert parse_amount("$1,875.40") == 1875.40
assert parse_amount("$734.01") == 734.01
print("parse_amount OK")

## 3. Parse the dates

Two date columns. `Transaction Date` is when the money was spent; `Posted Date` is when the account changed. They can differ by a few days. Which one you use depends on the question, and you'll choose deliberately in the exercises.


In [ ]:
from datetime import datetime

def parse_date(raw: str):
    return datetime.strptime(raw.strip(), "%m/%d/%Y").date()

print(parse_date("06/05/2026"))

## 4. Recover a missing merchant (regex comes back)

When the bank couldn't identify the vendor, `Merchant name` is blank and the name is buried in `Full description`. This reuses what you already know, `re.compile`, `.match`, `.group()`, with one new piece: the **capture group**.

Last lesson `m.group()` returned the whole match. Here we wrap the vendor part of the pattern in parentheses, and `m.group(1)` returns *just that captured piece*. `.match` anchors at the start of the string, which is why it works here, the vendor sits at the front.

It's a best-effort guess and it won't always succeed, which is itself the lesson about messy text.


In [ ]:
import re

MERCHANT_PATTERN = re.compile(r"([A-Z][A-Za-z&' ]+?)\s+\d")

def extract_merchant(description: str) -> str:
    m = MERCHANT_PATTERN.match(description)
    return m.group(1).strip() if m else description

# works cleanly here:
print(extract_merchant("CLOUDTUNES 06-09 888-555-0170 CA DEBIT CARD RECURRING PYMT"))
# defeated by the '*' before any digit, so it returns the whole string:
print(extract_merchant("ZIPRIDE *TRIP 06-03 HELP.ZIPRIDE.COM CA DEBIT CARD PURCHASE"))

## 5. Load everything into clean records

Now combine the pieces into one list of dicts, the tidy shape you'd otherwise have had to build by hand from raw text. The CSV reader gets us here almost for free.


In [ ]:
def load_transactions(path):
    records = []
    with open(path, newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            records.append({
                "posted_date": parse_date(row["Posted Date"]),
                "txn_date": parse_date(row["Transaction Date"]),
                "type": row["Transaction Type"],
                "description": row["Full description"],
                "merchant": row["Merchant name"] or extract_merchant(row["Full description"]),
                "category": row["Category name"],
                "subcategory": row["Sub-category name"],
                "amount": parse_amount(row["Amount"]),
                "daily_balance": parse_amount(row["Daily Posted Balance"]),
            })
    return records

records = load_transactions("statement.csv")
print("loaded", len(records), "transactions")
records[:2]

## Exercise 1 — Confirm the sign convention

Never trust the sign blindly: verify it against a transaction you recognize. Debits should be negative, credits positive.


In [ ]:
debits  = [r for r in records if r["amount"] < 0]
credits = [r for r in records if r["amount"] > 0]
print("debits :", len(debits), "  total", round(sum(r["amount"] for r in debits), 2))
print("credits:", len(credits), "  total", round(sum(r["amount"] for r in credits), 2))
print("net change over the month:", round(sum(r["amount"] for r in records), 2))

## Exercise 2 — Spending by category

Total spending per category. **Use `txn_date`, not `posted_date`** — the question is about *when money was spent*, which is the transaction date. (Write a sentence for yourself on why before moving on.)

Spending is money out, so we look at negative amounts and report them as positive totals.


In [ ]:
from collections import defaultdict

spend = defaultdict(float)
for r in records:
    if r["amount"] < 0:
        spend[r["category"]] += -r["amount"]

for cat, total in sorted(spend.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{cat:20s} {total:8.2f}")

## Exercise 3 — Reconcile a day's balance

`Daily Posted Balance` is the *end-of-day* balance, repeated on every row of that posted date (look at 06/05, three rows, one balance). Confirm it: a day's closing balance should equal the previous day's closing plus the sum of that day's amounts.


In [ ]:
from collections import OrderedDict

# closing balance stamped on each posted date
closing = OrderedDict()
day_total = OrderedDict()
for r in sorted(records, key=lambda r: r["posted_date"]):
    closing[r["posted_date"]] = r["daily_balance"]
    day_total.setdefault(r["posted_date"], 0.0)
    day_total[r["posted_date"]] = round(day_total[r["posted_date"]] + r["amount"], 2)

prev = None
for day, close in closing.items():
    if prev is None:
        print(day, "closing", close, "(first day)")
    else:
        expected = round(prev + day_total[day], 2)
        flag = "OK" if abs(expected - close) < 0.005 else "MISMATCH"
        print(day, "closing", close, " expected", expected, flag)
    prev = close

## Exercise 4 — How often does the merchant fallback work?

Find rows where the bank left `Merchant name` blank, run `extract_merchant` on the description, and judge whether the guess is usable. You'll see it works on some formats and gets defeated by others (anything with punctuation before the first digit).


In [ ]:
blanks = []
with open("statement.csv", newline="", encoding="utf-8-sig") as f:
    for row in csv.DictReader(f):
        if not row["Merchant name"]:
            blanks.append(row["Full description"])

print(len(blanks), "rows had no merchant name\n")
for desc in blanks:
    print(f"{extract_merchant(desc):25s} <-- {desc[:45]}")

## Exercise 5 — The near-duplicate trap

Two coffees on 06/10, two rideshares on 06/14: same merchant, same category, same day, differing only by amount (and a reference detail). They are all real. A naive dedup keyed on (date, merchant, category) silently deletes a genuine transaction. Watch it happen, then decide what key actually distinguishes them.


In [ ]:
seen = set()
kept, dropped = [], []
for r in records:
    key = (r["txn_date"], r["merchant"], r["category"])  # naive, lossy
    if key in seen:
        dropped.append(r)
    else:
        seen.add(key)
        kept.append(r)

print("kept:", len(kept), " dropped:", len(dropped))
print("\nthese real transactions would vanish:")
for r in dropped:
    print(" ", r["txn_date"], r["merchant"], r["amount"])